# Notebook 00 — Descarga de Datos Crudos

## Objetivo

Descarga los tres datasets crudos del proyecto y los guarda en `data/raw/`.

**Idempotencia**: `download_sp500()` y `download_macro()` verifican si el CSV ya existe
en `data/raw/` antes de pegarle a la API. Si ya está descargado, simplemente lo cargan.
Esto significa que correr este notebook de nuevo **no vuelve a descargar nada** a menos
que borres los archivos de `data/raw/` manualmente.

In [6]:
import sys
import os
sys.path.insert(0, "..")

from src import data_loader, utils

utils.set_plot_style()

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

START = "2008-01-01"
END   = "2025-12-20"

In [13]:
news_check = data_loader.load_news(RAW_DIR)
print(f"Noticias desde:    {news_check['Date'].min().date()}")
print(f"Noticias hasta:    {news_check['Date'].max().date()}")
print(f"Total de noticias: {len(news_check):,}")

Noticias desde:    2008-01-02
Noticias hasta:    2025-12-18
Total de noticias: 27,212


## Descarga de precios S&P 500

Ticker `^GSPC` via `yfinance`. Guarda en `data/raw/sp500_raw.csv`.

In [8]:
sp500_path = os.path.join(RAW_DIR, "sp500_raw.csv")
sp500 = data_loader.download_sp500(START, END, sp500_path)

print(f"S&P 500: {sp500.shape} | {sp500.index[0].date()} → {sp500.index[-1].date()}")

[data_loader] sp500_raw.csv ya existe en ../data/raw/sp500_raw.csv, se carga sin descargar.
S&P 500: (4522, 5) | 2008-01-02 → 2025-12-19


## Descarga de indicadores macroeconómicos (FRED)

Series descargadas: VIX (`VIXCLS`), spread de tasas (`T10Y2Y`), Fed Funds Rate
(`FEDFUNDS`), inflación (`CPIAUCSL`) y desempleo (`UNRATE`).

La API key se lee de `.env` dentro de `download_macro()` — no se expone en el notebook.
Guarda en `data/raw/macro_fred.csv`.

In [12]:
macro_path = os.path.join(RAW_DIR, "macro_fred.csv")
macro = data_loader.download_macro(START, END, macro_path)

print(f"Macro FRED: {macro.shape} | {macro.index[0].date()} → {macro.index[-1].date()}")

[data_loader] macro_fred.csv ya existe en ../data/raw/macro_fred.csv, se carga sin descargar.
Macro FRED: (4750, 5) | 2008-01-01 → 2025-12-19


## Noticias financieras (Kaggle)

Este dataset **no se descarga via API** — es un CSV estático de Kaggle (19,127 headlines,
2008–2024) que debe colocarse manualmente en `data/raw/sp500_news.csv`. Por eso no existe
un `download_news()` en `data_loader.py`, solo `load_news()`.

Si el archivo no está presente, hay que bajarlo de Kaggle antes de continuar.

In [10]:
news_path = os.path.join(RAW_DIR, "sp500_news_full.csv")
if not os.path.exists(news_path):
    raise FileNotFoundError(
        f"No se encontró {news_path}.\n"
        "Este dataset viene de Kaggle y no se descarga via API: "
        "hay que colocarlo manualmente en data/raw/sp500_news_full.csv."
    )

news = data_loader.load_news(RAW_DIR)
print(f"Noticias: {news.shape} | {news['Date'].min().date()} → {news['Date'].max().date()}")

Noticias: (27212, 2) | 2008-01-02 → 2025-12-18


## Resumen estadístico de los datos crudos

Verificación rápida de tipos de datos, nulos y shape de los tres datasets descargados.

In [ ]:
utils.dataset_summary("S&P 500 Precios", sp500)
utils.dataset_summary("Macro FRED", macro)
utils.dataset_summary("Noticias Financieras", news)


  S&P 500 Precios  |  4,522 filas  x  5 columnas


,dtype,non_null,null,null_%,unique
Close,float64,4522,0,0.0,4499
High,float64,4522,0,0.0,4486
Low,float64,4522,0,0.0,4494
Open,float64,4522,0,0.0,4490
Volume,int64,4522,0,0.0,4454



── Muestra aleatoria (5 filas) ──


,Close,High,Low,Open,Volume
Date,,,,,
2023-07-14,4505.419922,4527.759766,4499.560059,4514.609863,3647450000
2017-06-28,2440.689941,2442.969971,2428.020020,2428.699951,3479980000
2025-12-08,6846.509766,6878.270020,6827.189941,6875.200195,4757130000
2017-04-27,2388.770020,2392.100098,2382.679932,2389.699951,4099940000
2024-03-20,5224.620117,5226.189941,5171.549805,5181.689941,4064850000



  Macro FRED  |  4,750 filas  x  5 columnas


...

,vix,t10y2y,fedfunds,cpi,unrate
Date,,,,,
2022-06-10,27.75,0.09,1.21,294.957,3.6
2012-01-26,18.57,1.74,0.08,227.842,8.3
2009-02-12,41.25,1.86,0.22,212.705,8.3
2015-09-25,23.62,1.47,0.14,237.498,5.0
2014-06-10,10.99,2.19,0.10,237.231,6.1



  Noticias Financieras  |  27,212 filas  x  2 columnas


,dtype,non_null,null,null_%,unique
Title,str,27212,0,0.0,27109
Date,datetime64[us],27212,0,0.0,4092



── Muestra aleatoria (5 filas) ──


,Title,Date
19829,A Look Back at Gas and Liquid Handling Stocks'...,2024-07-20
15342,Michael Burry’s Scion Asset Management appears...,2023-08-15
8611,Synthetic ETFs see resurgence as investors rea...,2020-09-29
1109,Waterproof mobiles make a splash,2012-02-28
14297,"S&P 500, Nasdaq end at 9-month highs on econom...",2023-06-01


### Limpieza
El dataset de noticias financieras y el de los precios del S&P 500 no presentaron valores nulos. Por otro lado, el Macro FRED presentó 201 correspondientes a días sin dato en VIX, los cuales serán eliminados con `dropna()`. 